# 02 — Synthetic Error Insertion on FinQA (First 5 Rows)

**Goal:** Use the Groq API (`gemma2-9b-it`) to insert exactly one synthetic factual error  
into the `response` field of the first 5 rows of the FinQA training split.

**Error types:** Temporal · Numerical · Entity · Relation · Contradictory · Unverifiable

**Tag format:**
- Span-level: `<type><delete>original_span</delete><mark>corrupted_span</mark></type>`
- Sentence-level (Contradictory / Unverifiable): `<type>corrupted_sentence</type>`

> **Setup:** Paste your Groq API key into `FRED/.env` as shown below — no other config needed.
> ```
> GROQ_API_KEY=gsk_...
> ```


In [ ]:
# ── Install dependencies if needed ────────────────────────────────────────
# %pip install -q datasets groq python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from datasets import load_dataset
from groq import Groq

# ── Load API key from FRED/.env ───────────────────────────────────────────
# .env lives one level above notebooks/
env_path = Path(os.getcwd()).parent / ".env"
loaded   = load_dotenv(dotenv_path=env_path)
print(f".env found at : {env_path}  (loaded={loaded})")

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY or GROQ_API_KEY == "paste_your_groq_key_here":
    raise ValueError(
        "GROQ_API_KEY missing or not set.\n"
        "Open FRED/.env and replace 'paste_your_groq_key_here' with your key."
    )

# ── Config ────────────────────────────────────────────────────────────────
MODEL  = "gemma2-9b-it"
N_ROWS = 5

ERROR_TYPES = [
    "Temporal",
    "Numerical",
    "Entity",
    "Relation",
    "Contradictory",
    "Unverifiable",
]

client = Groq(api_key=GROQ_API_KEY)
print(f"✅ Groq client ready  |  model: {MODEL}")

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
ds    = load_dataset("rungalileo/ragbench", "finqa", trust_remote_code=True)
train = ds["train"]

print(f"FinQA train split: {len(train):,} rows")
print(f"Columns          : {train.column_names}")

In [ ]:
# ── Prompt template ───────────────────────────────────────────────────────
PROMPT_TEMPLATE = """\
You are a dataset-annotation assistant. Your task is to insert exactly ONE factual error \
into the RESPONSE below. Use the reference DOCUMENTS and QUESTION only to understand context \
— do NOT alter them.

Choose ONE error type at random from this list:
  Temporal, Numerical, Entity, Relation, Contradictory, Unverifiable

Tagging rules (follow exactly, no exceptions):
  • Span-level errors (Temporal / Numerical / Entity / Relation):
      <TYPE><delete>original_span</delete><mark>corrupted_span</mark></TYPE>
    Replace TYPE with the chosen error type in the exact casing shown above.
  • Sentence-level errors (Contradictory / Unverifiable):
      <TYPE>corrupted_sentence</TYPE>
    The corrupted sentence must replace the original sentence entirely inside the tag.

Constraints:
  - Insert EXACTLY one error. No more.
  - Output ONLY the tagged corrupted response. No explanation, no preamble.
  - Do NOT change any part of the response that is not involved in the error.
  - Do NOT add new sentences. Only corrupt or replace an existing span or sentence.

---
DOCUMENTS:
{documents}

QUESTION:
{question}

RESPONSE:
{response}
---

Output the tagged corrupted response now:"""

print("Prompt template defined ✓")

In [ ]:
# ── Helper: format documents list → readable string ───────────────────────
def format_documents(docs):
    """Join a list of document strings with separators, or pass through a plain string."""
    if isinstance(docs, list):
        return "\n\n".join(f"[Doc {i+1}] {d}" for i, d in enumerate(docs))
    return str(docs)  # already a string


# ── Helper: call Groq API ─────────────────────────────────────────────────
def insert_error(documents_raw, question, response):
    prompt = PROMPT_TEMPLATE.format(
        documents=format_documents(documents_raw),
        question=question,
        response=response,
    )
    chat_completion = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,   # randomness so error type varies across rows
        max_tokens=1024,
    )
    return chat_completion.choices[0].message.content.strip()


print("Helper functions defined ✓")

In [ ]:
# ── Main loop: process first N_ROWS ──────────────────────────────────────
results = []

for idx in range(N_ROWS):
    row = train[idx]

    # ── Extract fields (handle both common naming conventions) ─────────────
    documents = row.get("documents") or row.get("document") or row.get("context") or ""
    question  = row.get("question")  or row.get("query")    or ""
    response  = row.get("response")  or row.get("answer")   or row.get("output") or ""

    print(f"\n{'='*70}")
    print(f"ROW {idx+1} / {N_ROWS}")
    print(f"{'='*70}")
    print(f"\n📄 QUESTION:\n{question}")
    print(f"\n✅ ORIGINAL RESPONSE:\n{response}")

    corrupted = insert_error(documents, question, response)

    print(f"\n⚠️  CORRUPTED RESPONSE (tagged):\n{corrupted}")

    results.append({
        "idx"      : idx,
        "question" : question,
        "response" : response,
        "corrupted": corrupted,
    })

print(f"\n\n✔ Done — {len(results)} rows processed.")

In [ ]:
# ── Sanity check — confirm tags are present in every output ───────────────
import re

TAG_PATTERN = re.compile(
    r"<(Temporal|Numerical|Entity|Relation|Contradictory|Unverifiable)>"
)

print(f"{'Row':<5} {'Tag found':<15} {'Error type detected'}")
print("-" * 42)
for r in results:
    match = TAG_PATTERN.search(r["corrupted"])
    found = bool(match)
    etype = match.group(1) if match else "— NONE DETECTED —"
    print(f"{r['idx']+1:<5} {'✅' if found else '❌':<15} {etype}")